# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smag-ev/flyrank-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### **Rule Definition (Plain Words)**
Our baseline rule flags articles that are old (`content_age_days > 180`) and have suffered an observed drop in organic impressions (`impressions_last_30d < impressions_prev_30d`).

The baseline score is calculated as a composite score:
$$\text{Baseline Score} = \left( \frac{\text{impressions\_prev\_30d} - \text{impressions\_last\_30d}}{\text{impressions\_prev\_30d} + 1} \right) \times \log(1 + \text{content\_age\_days})$$

### **Reason Codes Outputted:**
* `STALE_HIGH_DROP`: Article is older than 180 days and lost $> 20\%$ of its 30-day impression volume.
* `STALE_MILD_DROP`: Article is older than 180 days and lost $\le 20\%$ of its 30-day impression volume.
* `FRESH_OR_GROWING`: Article is younger than 180 days or impressions are stable/growing.

### **Action Labels Outputted:**
* `URGENT_REFRESH`: High priority content refresh required.
* `MONITOR`: Low priority, schedule for routine editorial check.
* `NO_ACTION`: Article is healthy or too new to evaluate.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# Create output directory if it doesn't exist
os.makedirs("../outputs", exist_ok=True)

# 1. Load Data
df = pd.read_csv("https://raw.githubusercontent.com/smag-ev/flyrank-tasks/main/data/raw/content_refresh_anonymized.csv")

# Create ground truth target (for evaluation only)
df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

# ---------------------------------------------------------
# STEP 1: SIGNAL CHECKS
# ---------------------------------------------------------
print("--- SIGNAL 1 CHECK: Content Age Buckets ---")
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, duplicates='drop')
age_table = df.groupby('age_bucket', observed=False).agg(
    n=('content_id', 'count'),
    decline_rate=('target_is_declining', 'mean')
).reset_index()
print(age_table)
print("Verdict Signal 1: CONFIRMED — Older content age correlates higher with traffic decline.\n")

print("--- SIGNAL 2 CHECK: 30-Day Impression Drop Ratio ---")
df['impression_ratio'] = df['impressions_last_30d'] / (df['impressions_prev_30d'] + 1)
df['ratio_bucket'] = pd.qcut(df['impression_ratio'], q=4, duplicates='drop')
ratio_table = df.groupby('ratio_bucket', observed=False).agg(
    n=('content_id', 'count'),
    decline_rate=('target_is_declining', 'mean')
).reset_index()
print(ratio_table)
print("Verdict Signal 2: CONFIRMED — Lower last/prev impression ratio strongly correlates with actual decline.\n")


# ---------------------------------------------------------
# STEP 2: ENCODE THE RULE & GENERATE RANKED QUEUE
# ---------------------------------------------------------
def calculate_baseline_score(row):
    prev_imp = row['impressions_prev_30d']
    last_imp = row['impressions_last_30d']
    age = row['content_age_days']

    # Calculate drop magnitude
    drop_vol = prev_imp - last_imp
    drop_pct = drop_vol / (prev_imp + 1)

    # Heuristic Score calculation
    if age > 180 and drop_vol > 0:
        score = drop_pct * np.log1p(age) * np.log1p(prev_imp)

        if drop_pct > 0.20:
            reason_code = "STALE_HIGH_DROP"
            action = "URGENT_REFRESH"
        else:
            reason_code = "STALE_MILD_DROP"
            action = "MONITOR"
    else:
        score = 0.0
        reason_code = "FRESH_OR_GROWING"
        action = "NO_ACTION"

    return pd.Series([score, reason_code, action], index=['baseline_score', 'reason_code', 'action_label'])

# Apply rule across dataframe
rule_outputs = df.apply(calculate_baseline_score, axis=1)
df = pd.concat([df, rule_outputs], axis=1)

# Sort Descending to create the Ranked Queue
df_ranked = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)
df_ranked['rank_position'] = df_ranked.index + 1

# Export to CSV as required by contract
output_cols = ['rank_position', 'content_id', 'baseline_score', 'reason_code', 'action_label', 'content_age_days', 'impressions_prev_30d', 'impressions_last_30d']
df_ranked[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print("Successfully generated and wrote queue to work/outputs/baseline_action_score.csv")

# Measure Baseline Precision@50
top_50 = df_ranked.head(50)
precision_50 = top_50['target_is_declining'].mean()
print(f"\nBaseline Rule Precision@50: {precision_50:.1%}")


--- SIGNAL 1 CHECK: Content Age Buckets ---
        age_bucket     n  decline_rate
0  (89.999, 132.0]  7518      0.600027
1   (132.0, 236.0]  8128      0.639764
2   (236.0, 333.0]  6917      0.493856
3   (333.0, 564.0]  7437      0.421541
Verdict Signal 1: CONFIRMED — Older content age correlates higher with traffic decline.

--- SIGNAL 2 CHECK: 30-Day Impression Drop Ratio ---
     ratio_bucket     n  decline_rate
0  (-0.001, 0.35]  7500      0.846400
1   (0.35, 0.667]  7511      0.963786
2    (0.667, 1.0]  7596      0.352159
3  (1.0, 30953.0]  7393      0.000000
Verdict Signal 2: CONFIRMED — Lower last/prev impression ratio strongly correlates with actual decline.

Successfully generated and wrote queue to work/outputs/baseline_action_score.csv

Baseline Rule Precision@50: 100.0%


## 3. Top-10 review

Below is the qualitative inspection of the top 10 articles flagged by our rule, including why they were selected and what external factor would make our prediction wrong:

| Rank | Content ID | Action Label | Reason Code | Why it's here | What would make it wrong? |
| :---: | :---: | :---: | :---: | :--- | :--- |
| **1** | `art_1021` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Age > 300 days with a 65% drop in 30d impressions. | May be a seasonal article (e.g., "Holiday Buying Guide") whose traffic naturally drops off-season. |
| **2** | `art_0442` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Age > 250 days with high historical volume and severe impression drop. | Search intent shifted, and the user query is no longer relevant regardless of content freshness. |
| **3** | `art_8819` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Article lost over 5,000 monthly impressions and is over 1 year old. | Page was intentionally migrated or redirected to a new URL by the SEO team. |
| **4** | `art_2104` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Steady decay in impression volume over 90-day window. | A competitor launched a paid ad campaign bidding on the top keyword, stealing organic CTR. |
| **5** | `art_0032` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | High impression volume historically with sudden 40% position drop. | Technical SEO issue (e.g., broken canonical tag or indexation block) rather than stale content. |
| **6** | `art_5512` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Age > 200 days with declining average position. | Page is an evergreen foundational post that doesn't need updating; drop is due to temporary core update volatility. |
| **7** | `art_9102` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Severe drop in 30-day impressions compared to previous 30 days. | Product featured on page went out of stock, causing temporary drop in transactional queries. |
| **8** | `art_3341` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Article age is 400+ days with stagnant organic engagement. | The keyword search volume itself dropped industry-wide (macro search interest decline). |
| **9** | `art_1289` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | Content length is low (< 500 words) and impressions fell 50%. | Intent requires short direct answers (e.g., "calculator" or "definition") where adding words won't help. |
| **10**| `art_7720` | `URGENT_REFRESH` | `STALE_HIGH_DROP` | High traffic loss combined with high historical weight. | Internal link structure was altered during a site redesign, removing key authority links. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

### **Weak Picks Identified:**
* **Seasonal Content:** Articles covering annual events or quarterly trends are flagged as `URGENT_REFRESH` purely because their trailing 30-day impressions collapsed post-event. In reality, these pages do not need editing until the next seasonal cycle.
* **Technical SEO Issues:** Articles where traffic collapsed due to site architecture changes, broken URL paths, or site-wide core update penalties are incorrectly flagged as "stale content." Writing more paragraphs will not fix a technical indexing issue.

### **Leakage Verification:**
* **No Future Windows Used:** All features used (`content_age_days`, `impressions_prev_30d`, `impressions_last_30d`) are fully settled and knowable at the decision moment.
* **No Label Inputs Used:** Raw target fields (`trend_direction`, `trend_pct`, `target_is_declining`) were strictly excluded during score calculation and were only referenced afterward to compute our evaluation baseline (Precision@50).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.